# Pilot run — Qwen2-Audio-7B-Instruct (Kaggle)

Runs the full pilot after the smoke test has passed:

1. **Main pilot**: 15 fictitious facts + 8 knowledge-neutral control facts, 8 conditions, conflict items in both A/B orders.
2. **Complexity dataset**: 12 fictitious facts at 4 sentence-difficulty levels.

**Notebook settings (right sidebar):** Accelerator **GPU T4 x2**, Internet **On**, and for a private repo the `GITHUB_TOKEN` secret.
Total: 710 stimuli, each with one forward pass plus one short greedy generation.

## 1. GPU check

In [ ]:
import os, subprocess, torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"
gpus = [torch.cuda.get_device_properties(i) for i in range(torch.cuda.device_count())]
VRAM_GB = sum(g.total_memory for g in gpus) / 1e9
for g in gpus:
    print(f"{g.name}: {g.total_memory / 1e9:.1f} GB")
# 7B in fp16 needs ~16 GB of weights plus activations; device_map="auto" splits it across GPUs
QUANTIZATION = "none" if VRAM_GB >= 20 else "4bit"
# native bf16 needs compute capability >= 8.0 (A100/L4); T4 / P100 only emulate it -> float16
DTYPE = "bfloat16" if min(torch.cuda.get_device_capability(i)[0] for i in range(len(gpus))) >= 8 else "float16"
print(f"total {VRAM_GB:.1f} GB -> quantization: {QUANTIZATION} | dtype: {DTYPE}")

## 2. Get the project code

Clones into `/kaggle/working/Thesis`, so `results/` ends up in the notebook's **Output** tab.
Set `REPO_URL` to your GitHub repo. For a private repo the token is read from Kaggle Secrets (never paste it into the notebook).

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # <- edit
BRANCH = "main"
REPO = "/kaggle/working/Thesis"

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
    print("using GITHUB_TOKEN from Kaggle Secrets")
except Exception:
    print("no GITHUB_TOKEN secret -> cloning as a public repo")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)   # pick up new pushes
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
assert os.path.exists("src/eval/run_model.py"), f"src/ not found under {REPO}"
print("working in", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 3. Install dependencies
Kaggle already has `torch` and `transformers`; this adds `bitsandbytes`, `edge-tts`, etc.

In [ ]:
!pip install -q -r requirements.txt
import transformers; print("transformers", transformers.__version__)

## 4. Kaggle config
Copies `configs/pilot.yaml` with the quantization / dtype chosen for this GPU.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/pilot.yaml"))
cfg["model"]["quantization"] = QUANTIZATION
cfg["model"]["dtype"] = DTYPE
yaml.safe_dump(cfg, open("configs/kaggle.yaml", "w"), sort_keys=False)
print(yaml.safe_dump(cfg["model"]), "eval:", cfg["eval"]["uq"], "| greedy_check:", cfg["eval"]["greedy_check"])

## 5. Main pilot: 15 fictitious + 8 control facts

230 stimuli (conflict items in both A/B orders). Resumable: re-run the cell after a crash.

In [ ]:
!python -m src.eval.run_model --config configs/kaggle.yaml --out results/pilot_run_1

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/pilot_run_1/summary.md").read()))

In [ ]:
for kind in ["fictitious", "control"]:
    print(f"===== {kind}")
    !python -m src.analysis.h1_h2_h3 --results results/pilot_run_1 --kind {kind}
display(Image("results/pilot_run_1/audio_conf_vs_conflict_fictitious.png"))

## 6. Complexity dataset: 12 facts x 4 levels (L0 baseline, L1 extra info, L2 embedding, L3 lure)

480 stimuli. The key question: does the conflict text-follow rate rise as audio-only confidence falls?

In [ ]:
!python -m src.eval.run_model --config configs/kaggle.yaml --stimuli data/complexity/stimuli.jsonl --out results/complexity_run_1

In [ ]:
!python -m src.analysis.complexity_sweep --results results/complexity_run_1
display(Image("results/complexity_run_1/complexity_sweep.png"))

In [ ]:
display(Markdown(open("results/complexity_run_1/summary.md").read()))

## 7. Keep the results

Everything under `/kaggle/working` is in the **Output** tab; **Save Version → Save & Run All** keeps it permanently.

In [ ]:
import shutil
for run in ["pilot_run_1", "complexity_run_1"]:
    shutil.make_archive(f"/kaggle/working/{run}", "zip", f"results/{run}")
print("download: Output tab -> pilot_run_1.zip, complexity_run_1.zip")